# Trial Pipeline Health Summary — SQL

SQL-cell equivalent of `trial_pipeline_health_summary.ipynb`, using
`%%sql` cells instead of the DataFrame API. Same result: reads the
`clinical_trials.trial_registry` bronze table and persists
`clinical_trials.trial_pipeline_health_summary_v1`.

Requires the bronze table to already exist — run
`data-generators/clinical_trials_registry_ingest.ipynb` first.

`%%sql` here follows Databricks-style notebook cell magic. If running
outside a platform that provides it natively, install and load
`sparksql-magic` first: `pip install sparksql-magic` then
`%load_ext sparksql_magic` (bound to the `spark` session created below).

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## Transform + persist: pipeline health by phase

In [ ]:
%%sql
CREATE OR REPLACE TABLE clinical_trials.trial_pipeline_health_summary_v1 AS
SELECT
    phase,
    COUNT(*) AS trial_count,
    ROUND(AVG(enrollment_count), 1) AS avg_enrollment,
    SUM(CASE WHEN overall_status = 'TERMINATED' THEN 1 ELSE 0 END) AS terminated_count,
    SUM(CASE WHEN overall_status = 'COMPLETED' THEN 1 ELSE 0 END) AS completed_count,
    SUM(CASE WHEN overall_status IN ('RECRUITING', 'NOT_YET_RECRUITING') THEN 1 ELSE 0 END) AS active_recruiting_count,
    ROUND(SUM(CASE WHEN overall_status = 'TERMINATED' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS termination_rate_pct
FROM clinical_trials.trial_registry
GROUP BY phase
ORDER BY trial_count DESC

## Verify

In [ ]:
%%sql
SELECT COUNT(*) AS total_phases FROM clinical_trials.trial_pipeline_health_summary_v1

In [ ]:
%%sql
SELECT * FROM clinical_trials.trial_pipeline_health_summary_v1
ORDER BY trial_count DESC